# 02 — Data & Training: Datasets, DataLoaders, Losses, Optimizers, Schedulers, AMP

Goal: train models correctly, efficiently, and reproducibly.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

In [ ]:

def seed_all(seed=1234):
    import random, numpy as np, torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_all(42)

## 1. Dataset and DataLoader fundamentals

- implement custom Dataset
- DataLoader batching, shuffling, workers
- custom `collate_fn` for variable shapes

In [ ]:

from torch.utils.data import Dataset, DataLoader
import torch

class ToyDataset(Dataset):
    def __init__(self, n=2000, d=5):
        self.X = torch.randn(n, d)
        true_w = torch.randn(d, 1)
        self.y = (self.X @ true_w + 0.1*torch.randn(n,1)).squeeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

ds = ToyDataset()
dl = DataLoader(ds, batch_size=64, shuffle=True, num_workers=0)
xb, yb = next(iter(dl))
xb.shape, yb.shape

## 2. Reusable training loop template

Includes:
- train/eval modes
- inference_mode eval
- gradient clearing
- metric hooks

In [ ]:

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class MLPRegressor(nn.Module):
    def __init__(self, d=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): 
        return self.net(x).squeeze(-1)

def train_epoch(model, dl, optimizer, loss_fn, device, max_grad_norm=None):
    model.train()
    total, n = 0.0, 0
    for xb, yb in dl:
        xb = xb.to(device); yb = yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        total += loss.item() * xb.size(0); n += xb.size(0)
    return total / n

@torch.inference_mode()
def eval_epoch(model, dl, loss_fn, device):
    model.eval()
    total, n = 0.0, 0
    for xb, yb in dl:
        xb = xb.to(device); yb = yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        total += loss.item() * xb.size(0); n += xb.size(0)
    return total / n

model = MLPRegressor(d=5).to(device)
opt = optim.AdamW(model.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()

for epoch in range(5):
    tr = train_epoch(model, dl, opt, loss_fn, device, max_grad_norm=1.0)
    va = eval_epoch(model, dl, loss_fn, device)
    print(epoch, tr, va)

## 3. Loss functions (must-know)

- regression: MSE, SmoothL1
- classification: CrossEntropy (logits + class indices)
- multi-label: BCEWithLogits (logits + {0,1} targets)

In [ ]:

logits = torch.randn(4, 10, device=device)
labels = torch.randint(0, 10, (4,), device=device)
ce = nn.CrossEntropyLoss()(logits, labels)

logits2 = torch.randn(4, 5, device=device)
targets2 = torch.randint(0, 2, (4,5), device=device).float()
bce = nn.BCEWithLogitsLoss()(logits2, targets2)

float(ce), float(bce)

## 4. Optimizers and parameter groups

AdamW is a modern default; exclude biases and norms from weight decay.

In [ ]:

def param_groups_weight_decay(model, weight_decay=0.01):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim == 1 or name.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)
    return [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]

model = MLPRegressor(d=5).to(device)
opt = optim.AdamW(param_groups_weight_decay(model, 0.01), lr=3e-4)
[len(pg["params"]) for pg in opt.param_groups], [pg["weight_decay"] for pg in opt.param_groups]

## 5. Schedulers

Schedulers can be decisive for convergence and final quality.

In [ ]:

from torch.optim.lr_scheduler import StepLR

model = MLPRegressor(d=5).to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3)
sched = StepLR(opt, step_size=2, gamma=0.5)

for epoch in range(5):
    lr = opt.param_groups[0]["lr"]
    loss = train_epoch(model, dl, opt, nn.MSELoss(), device)
    sched.step()
    print("epoch", epoch, "lr", lr, "loss", loss)

## 6. AMP, grad accumulation, and clipping (CUDA)

AMP improves throughput on modern GPUs. Use GradScaler to avoid underflow.

In [ ]:

from torch.cuda.amp import autocast, GradScaler

use_amp = torch.cuda.is_available()
scaler = GradScaler(enabled=use_amp)

model = MLPRegressor(d=5).to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

accum_steps = 4
opt.zero_grad(set_to_none=True)

for i, (xb, yb) in enumerate(dl):
    xb = xb.to(device); yb = yb.to(device)
    with autocast(enabled=use_amp):
        pred = model(xb)
        loss = loss_fn(pred, yb) / accum_steps
    scaler.scale(loss).backward()
    if (i+1) % accum_steps == 0:
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
    if i > 10:
        break

print("AMP enabled:", use_amp)

## 7. Checkpointing correctly

Include model + optimizer + scheduler + scaler + metadata.

In [ ]:

import torch

ckpt = {
    "model_state": model.state_dict(),
    "opt_state": opt.state_dict(),
    "scaler_state": scaler.state_dict(),
    "epoch": 0,
}
torch.save(ckpt, "checkpoint_demo.pt")
loaded = torch.load("checkpoint_demo.pt", map_location="cpu")
list(loaded.keys())